# Colab 환경에서 T4로 작업하였습니다.
### 로컬에서 활용할 경우, file_path에 적절한 디렉토리를 지정해주세요

임베딩 전체 2시간 정도 소요되었습니다.

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

model = SentenceTransformer(
    "dragonkue/snowflake-arctic-embed-l-v2.0-ko"
)

## 여기서 원하는 경로로 수정하시면 됩니다.
file_path = "/content/drive/MyDrive/dontalk/rag_dataset.jsonl"

df = pd.read_json(file_path, lines=True)

# 빈 텍스트 제거
df["text"] = df["text"].fillna("").astype(str)
df = df[df["text"].str.strip().ne("")].reset_index(drop=True)

texts = df["text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=8,              # GPU 메모리에 따라 8, 16, 32 조절
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# 행 수와 벡터 수 확인
assert len(df) == len(embeddings)

print(embeddings.shape)


 Loading weights: 100% 391/391 [00:00<00:00, 6369.89it/s]Batches: 100% 10000/10000 [1:43:03<00:00,  2.50it/s](80000, 1024)

임베딩을 넘파이 파일로 저장 (코랩에서 저장 안하면 세션 날아갈때 같이 날아가요)

In [ ]:
embedding_path = "/content/drive/MyDrive/dontalk/rag_embeddings.npy"
np.save(embedding_path, embeddings.astype("float32"))

임베딩 데이터와 본래 데이터 (메타+본문)을 병합 

원래 무결성 검사를 진행해야 하지만 우리는 결측이 없는 데이터들이므로 PASS 했습니다.

In [ ]:
df["embedding"] = embeddings.tolist()

df.to_json(
    "/content/drive/MyDrive/dontalk/rag_dataset_with_embeddings.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

질의 쿼리는 다음과 같이 구성합니다.

```python
    query = "자동차 보험 가입 조건이 궁금합니다."

    query_embedding = model.encode(
        [query],
        prompt_name="query",
        normalize_embeddings=True
    )
```

# 실제 테스트 코드

데이터,임베딩 로드

In [ ]:
import numpy as np
import pandas as pd

dataset_path = "/content/drive/MyDrive/dontalk/rag_dataset.jsonl"
embedding_path = "/content/drive/MyDrive/dontalk/rag_embeddings.npy"

df = pd.read_json(dataset_path, lines=True)
document_embeddings = np.load(embedding_path).astype("float32")

assert len(df) == len(document_embeddings)

# 문서 임베딩 정규화
document_embeddings /= np.linalg.norm(
    document_embeddings,
    axis=1,
    keepdims=True
)

질의 함수(search_documents) 정의

In [ ]:

def search_documents(query, top_k=5):
    # 질의는 query prompt를 사용
    query_embedding = model.encode(
        [query],
        prompt_name="query",
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # 정규화된 벡터끼리는 내적 = cosine similarity
    scores = document_embeddings @ query_embedding

    top_indices = np.argsort(scores)[::-1][:top_k]

    result = df.iloc[top_indices].copy()
    result.insert(0, "score", scores[top_indices])

    return result.reset_index(drop=True)

모델 로드

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "dragonkue/snowflake-arctic-embed-l-v2.0-ko"
)

테스트

In [ ]:
query = "자동차 보험 가입 조건이 궁금합니다."
results = search_documents(query, top_k=5)

pd.set_option("display.max_colwidth", 500)

for i, row in results.iterrows():
    print(f"[{i + 1}위] score={row['score']:.4f}")
    print(row["text"])
    print("-" * 80)

### 출력문은 다음과 같습니다.

```plaintext
[1위] score=0.6785
요구사항: 고객의 자동차보험 가입 조건과 보장 내용을 상세히 안내하시오.
고객 질문: ●●일 보험을 가입하려고 합니다. 구체적인 보장 내용과 조건을 알려 주실 수 있나요?
상담사 답변: 요청하시는 차량 번호를 알려 주시면 정확한 보험 계약 절차를 진행할 수 있습니다. 보험 효력은 지정된 날짜부터 적용되며, 보장 범위와 조건을 상세히 안내드리겠습니다. 보장 제외 사항으로는 대리운전 시 발생하는 사고와 은행 관련 손해는 포함되지 않으며, 면허증 소지자만 운전 가능합니다. 추가 보험료는 ●●원이며, 납부는 카드 결제 또는 자동 이체로 가능합니다. 결제 방법에 따라 혜택이 달라질 수 있습니다.
꼬리 질문: 추가로 필요한 사항이 있나요?
종합 답변: 보험 가입을 위해 먼저 차량 번호를 알려 주시기 바랍니다. 알려 주신 번호를 기반으로 계약 절차를 진행하고, 보험 효력 시작일과 적용 기간을 확인해 드리겠습니다. 보장 범위는 기본적인 사고 보상과 차량 손해를 포함하지만, 대리운전 중 발생한 사고와 은행 관련 손해는 제외됩니다. 또한, 면허증을 소지한 운전자만 보장이 적용됩니다. 추가 보험료는 ●●원이며, 결제는 카드 결제 또는 자동 이체 중 선택하실 수 있습니다. 결제 방식에 따라 제공되는 혜택이 달라질 수 있으니 원하시는 방법을 알려 주시면 최종 안내해 드리겠습니다.
--------------------------------------------------------------------------------
[2위] score=0.6558
요구사항: 자동차보험 가입 절차와 필요 서류, 연령별 보험료 적용 기준을 안내하시오.
고객 질문: 자동차보험을 가입하려는데 절차가 어떻게 되나요?
상담사 답변: 보험 계약을 위해 차량 등록증 사본과 운전면허증이 필요합니다. 계약서를 이메일로 발송하고, 전자 서명을 통해 계약을 완료할 수 있습니다. 기본 책임보험 외에 자차 손해, 대물 상해 특약을 추가하실 수 있으며, 보장 범위와 비용을 상세히 안내드리겠습니다. 모바일 앱을 통해 간편하게 신청이 가능하며, 계약이 완료되면 보증서와 증권을 전자 형태로 1~2일 이내에 발송해 드립니다. 이미 납부하신 보험료에 대한 환급은 계약 해지 시 잔여 기간에 비례하여 계산됩니다.
꼬리 질문: 자동차보험에서 연령에 따른 보험료 차등 적용 기준을 알려 주세요.
종합 답변: 보험 계약을 위해 차량 등록증과 운전면허증 사본이 필요하며, 계약서는 이메일로 보내드리고 전자 서명으로 완료할 수 있습니다. 만 20세 이상부터 가입 가능하며, 보험료는 연령에 따라 다르게 적용되니 참고해 주시기 바랍니다.
--------------------------------------------------------------------------------
[3위] score=0.6546
요구사항: 자동차보험 가입 여부 확인 및 신규 가입 절차를 안내하시오.
고객 질문: 보험 가입 여부를 온라인 시스템을 통해 직접 확인할 수 있는지 궁금합니다. 만약 현재 가입이 되어 있지 않다면, 새로 보험에 가입하는 절차를 진행해야 할 것 같은데, 이 부분에 대해 안내 부탁드립니다.
상담사 답변: 고객님께서 요청하신 내용을 토대로 저희 시스템에서 고객님의 보험 가입 상태를 신속하게 확인해 드리겠습니다. 가입 여부뿐만 아니라 관련된 상세 정보도 함께 안내해 드릴 수 있으니 조금만 기다려 주시기 바랍니다.
꼬리 질문: 지금 상황을 보면 보험 가입이 어려운 상태인 것 같은데, 그 원인이나 이유에 대해 좀 더 구체적이고 자세하게 설명해 주실 수 있을까요?
종합 답변: 고객님께서는 온라인 시스템을 통해 현재 자동차보험 가입 여부를 확인하고자 하셨습니다. 확인 결과, 현재 고객님의 차량에 대한 보험이 미가입 상태임을 확인했습니다. 따라서 신규 자동차보험을 가입하셔야 합니다. 보험 가입이 어려운 이유는 신용정보원에서 제공하는 정보가 누락되었거나 확인되지 않았기 때문일 수 있습니다. 정확한 원인을 파악하기 위해서는 고객님의 신용조회 동의서와 최근 무사고 증명서를 제출해 주시고, 추가로 필요한 서류를 이메일로 보내주시면 검토 후 상세히 안내드리겠습니다. 이후 보험료 견적을 제공하고, 원하시는 특약을 선택하신 후 계약 절차를 진행하실 수 있습니다.
--------------------------------------------------------------------------------
[4위] score=0.6537
요구사항: 자동차보험 온라인 가입 절차와 필요 서류, 할인 적용 조건을 안내하시오.
고객 질문: 자동차보험을 인터넷으로 가입하려고 하는데, 온라인에서 간편하게 가입할 수 있는 방법이나 절차가 어떻게 되는지 자세히 알려주실 수 있나요?
상담사 답변: 보험료에 차이가 있을 수 있으니, 고객님께서 적용받으실 수 있는 할인 카드 종류와 그에 따른 조건을 꼼꼼히 확인해 드려서 가장 유리한 혜택을 받으실 수 있도록 안내해 드리겠습니다.
꼬리 질문: 그렇다면 인터넷으로 가입하는 것이 가능한지, 온라인 가입 절차는 어떻게 진행되는지 그리고 가입 시에 꼭 준비해야 하는 서류가 무엇인지도 자세히 설명해 주실 수 있나요?
종합 답변: 고객님께서는 인터넷을 통해 자동차보험에 가입하고자 하시는 것으로 확인되었습니다. 먼저, 온라인 가입이 가능한지 여부는 현재 보유하고 계신 할인 카드와 적용 가능한 프로모션에 따라 달라질 수 있습니다. 할인 카드 종류와 적용 조건을 확인하기 위해서는 고객님의 카드 번호와 가입 예정인 보험상품 정보를 제공해 주시면 정확히 안내해 드릴 수 있습니다. 
온라인으로 가입이 가능할 경우, 다음 절차를 따라 주시기 바랍니다. 1) 보험사의 공식 웹사이트 또는 모바일 앱에 접속합니다. 2) 자동차보험 메뉴에서 신규 가입을 선택하고, 차량 정보와 운전자 정보를 입력합니다. 3) 할인 카드 번호를 입력하여 적용 가능한 할인율을 확인합니다. 4) 견적서가 생성되면 보험료와 보장 내용을 검토하고 동의합니다. 5) 신분증 사본과 차량 등록증 사본을 전자문서 형태로 업로드합니다. 6) 결제 방법을 선택하고 온라인 결제를 완료합니다. 
필요 서류는 신분증(주민등록증 또는 운전면허증)과 차량 등록증이며, 전자형태로 사진이나 스캔본을 제출하시면 됩니다. 모든 절차가 완료되면 보험증권은 이메일 또는 모바일 앱을 통해 발급됩니다. 추가로 궁금하신 사항이 있으면 언제든지 문의해 주시기 바랍니다.
--------------------------------------------------------------------------------
[5위] score=0.6490
요구사항: 자동차보험 가입 절차를 안내하시오.
고객 질문: 자동차보험 가입 절차가 어떻게 되는지 궁금합니다.
상담사 답변: 자동차보험을 처음 신청하실 때는 먼저 기본 보장 내용과 가입 가능한 옵션을 안내받으시게 됩니다. 이후 보험료 산정을 위해 차량 번호와 차량 등록 증명서에 기재된 소유자 성함을 확인합니다. 차량 번호를 정확히 알려주시면 위험도 평가와 보험료 산정에 반영됩니다. 제공해 주신 정보를 토대로 계약서 초안을 작성하고, 신원 확인을 위해 주민등록번호와 연락 가능한 휴대폰 번호를 추가로 확인합니다. 모든 정보가 확인되면 녹취를 진행하고, 최종 계약 내용을 고객님께 안내드린 후 동의를 받습니다. 동의가 완료되면 계약이 체결되며, 계약서와 보험증권을 발송해 드립니다.
꼬리 질문: 직업이 직장인인데 보험료에 영향이 있나요?
종합 답변: 자동차보험 가입 시 기본 보장 내용과 선택 가능한 옵션을 안내해 드리며, 보험료 산정을 위해 차량 번호와 소유자 정보를 확인합니다. 이후 신원 확인과 계약서 작성 절차를 거쳐 최종 계약 내용을 안내해 드리고 동의를 받으면 계약이 완료되며, 계약서와 보험증권을 발송해 드립니다.
--------------------------------------------------------------------------------
```